# Ink Prediction Failure Atlas — walkthrough

Reproduces every headline number in the README from `data/atlas.csv` and
`data/labels/`. **No download, no GPU, no model** — the heavy step (scoring 46,764
windows of published ds8 JPEGs) already happened; this notebook re-derives the
conclusions from its output so you can check them without re-running it.

Needs only `numpy` (and `matplotlib` for the last two plots).

In [ ]:
import csv, os, collections
import numpy as np

ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else "."
D = lambda *p: os.path.join(ROOT, "data", *p)

def load(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))

atlas = load(D("atlas.csv"))
score = np.array([float(r["mass_letter"]) for r in atlas])
print(f"windows        : {len(atlas):,}")
print(f"predictions    : {len({(r['scroll'], r['segment'], r['recipe']) for r in atlas})}")
print(f"segments       : {len({(r['scroll'], r['segment']) for r in atlas})}")
print(f"scrolls        : {sorted({r['scroll'] for r in atlas})}")

## 1. Every window covers the same physical area

This is what makes "% of windows" mean "% of papyrus area", and it is the fix for the
third of three unit bugs described in the README. The window is defined in **microns**,
so its size in pixels differs per prediction — a fixed pixel window would cover 11× more
papyrus on the 7.91 µm scan than on the 2.4 µm ones, and a mass fraction over 11× the
area dilutes.

In [ ]:
seen = {}
for r in atlas:
    seen.setdefault(float(r["pitch_um"]), set()).add(int(r["window_px"]))

print(f"{'pitch um/px':>12}{'window px':>12}{'physical mm':>14}")
for pitch in sorted(seen):
    for wpx in sorted(seen[pitch]):
        print(f"{pitch:>12}{wpx:>12}{wpx * pitch / 1000:>14.2f}")
print("\nSame physical size everywhere (the 2488 px row is the 7.91 um scan).")

## 2. The score is not a brightness proxy

The obvious failure mode for a legibility metric is that it silently measures *amount of
ink*. An early version did: it correlated 0.773 with the fraction of pixels above
threshold, which is a brightness meter with extra steps. Making the component-size
criterion a **band** instead of a lower bound (a saturated white smear is one enormous
connected component, and scores perfectly under a lower bound) brought it down.

In [ ]:
frac_above = np.array([float(r["frac_above"]) for r in atlas])
median_ext = np.array([float(r["median_extent"]) for r in atlas])
smear = np.array([float(r["mass_smear"]) for r in atlas])

print(f"corr(score, frac_above)    : {np.corrcoef(score, frac_above)[0,1]:.3f}")
print(f"corr(score, median_extent) : {np.corrcoef(score, median_ext)[0,1]:.3f}")
print(f"corr(score, mass_smear)    : {np.corrcoef(score, smear)[0,1]:.3f}")
print("\nscore distribution:")
for q in (1, 10, 25, 50, 75, 90, 99):
    print(f"  p{q:<3} {np.percentile(score, q):.4f}")

## 3. Calibration against the blind labels

The labels were produced without seeing the score: the sheets carry an index and nothing
else, in shuffled order, and the index → window key was written first. See
`data/labels/README.md`.

**AUC is reported first because it has no threshold to tune.** A metric can always be made
to look good at *some* cutoff; it cannot be made to look good under AUC.

In [ ]:
key = load(D("labels", "labeling_key.csv"))
lab = {int(r["i"]): r["label"].strip() for r in load(D("labels", "labels.csv")) if r["label"].strip()}

joined = [(float(r["mass_letter"]), lab[int(r["i"])]) for r in key if int(r["i"]) in lab]
s = np.array([j[0] for j in joined])
y = [j[1] for j in joined]
print(f"{len(lab)} labels written, {len(joined)} join the current window grid")
print(dict(collections.Counter(y)))

print(f"\n{'label':<10}{'n':>5}{'mean':>8}{'p25':>8}{'median':>8}{'p75':>8}")
for k in ("text", "partial", "speckle"):
    v = s[[i for i, t in enumerate(y) if t == k]]
    print(f"{k:<10}{len(v):>5}{v.mean():>8.3f}{np.percentile(v,25):>8.3f}"
          f"{np.median(v):>8.3f}{np.percentile(v,75):>8.3f}")

In [ ]:
def auc(pos, neg):
    """Mann-Whitney U: P(a random positive outranks a random negative)."""
    allv = np.concatenate([pos, neg])
    order = allv.argsort()
    ranks = np.empty(len(allv), float)
    ranks[order] = np.arange(1, len(allv) + 1)
    _, inv, cnt = np.unique(allv, return_inverse=True, return_counts=True)
    sums = np.zeros(len(cnt)); np.add.at(sums, inv, ranks)   # average over ties
    ranks = (sums / cnt)[inv]
    return (ranks[:len(pos)].sum() - len(pos)*(len(pos)+1)/2) / (len(pos)*len(neg))

pos = s[[i for i, t in enumerate(y) if t == "text"]]
neg = s[[i for i, t in enumerate(y) if t != "text"]]
print(f"AUC (text vs rest) = {auc(pos, neg):.3f}    (0.5 = worthless, 1.0 = perfect)\n")

print(f"{'thresh':>7}{'TP':>5}{'FP':>5}{'FN':>5}{'prec':>7}{'recall':>8}{'F1':>7}")
best = None
for t in np.arange(0.80, 1.00, 0.025):
    tp, fp, fn = int((pos>=t).sum()), int((neg>=t).sum()), int((pos<t).sum())
    if tp == 0: continue
    p, rc = tp/(tp+fp), tp/(tp+fn); f1 = 2*p*rc/(p+rc)
    best = best if best and best[0] >= f1 else (f1, t, p, rc)
    print(f"{t:>7.3f}{tp:>5}{fp:>5}{fn:>5}{p:>7.3f}{rc:>8.3f}{f1:>7.3f}")
print(f"\nbest F1 = {best[0]:.3f} at {best[1]:.3f}  (precision {best[2]:.3f}, recall {best[3]:.3f})")
THRESH = round(best[1], 3)

## 4. The headline

Note the shape of the threshold sweep: the answer moves from 12.6% to 0.3% between 0.850
and 0.950. **The threshold is doing most of the work**, which is exactly why it has to be
calibrated against labels rather than picked off the distribution. An earlier version of
this project anchored it on two hand-verified zones from one segment of one scroll — the
scroll that then topped the resulting table.

In [ ]:
print(f"corpus of {len(score):,} equal-area windows:\n")
for t in (0.80, 0.85, 0.875, 0.90, 0.95):
    mark = "  <- calibrated" if abs(t - THRESH) < 1e-9 else ""
    print(f"  >= {t:.3f}   {int((score>=t).sum()):>6,}   {float((score>=t).mean()):>6.2%}{mark}")

by = {}
for r in atlas:
    d = by.setdefault(r["scroll"], [0, 0])
    d[0] += 1; d[1] += float(r["mass_letter"]) >= THRESH
print(f"\nper scroll at {THRESH}:")
print(f"{'scroll':<14}{'windows':>9}{'legible':>9}{'pct':>8}")
for k, d in sorted(by.items(), key=lambda kv: -kv[1][1]/kv[1][0]):
    print(f"{k:<14}{d[0]:>9,}{d[1]:>9}{d[1]/d[0]:>8.2%}")
print("\nCaveat: PHercParis4 is 68% of all windows. Per-scroll figures are comparable to")
print("each other; the global average is de facto a PHercParis4 average.")

## 5. Where it fails — the actionable output

A percentage says how much of a scroll is readable, not *where*. Predictions with **zero**
readable windows are the failure list; `data/hotspots.csv` and `data/segment_summary.csv`
carry full-resolution coordinates, so a fix has a before/after number instead of an opinion.

In [ ]:
summ = load(D("segment_summary.csv"))
ranked = [r for r in summ if int(r["windows"]) >= 20]   # a 1-window prediction at 100% is noise
dead = [r for r in ranked if int(r["n_text"]) == 0]
print(f"{len(dead)} of {len(ranked)} ranked predictions ({len(dead)/len(ranked):.1%}) "
      f"have ZERO readable windows\n")
for k, n in collections.Counter(r["scroll"] for r in dead).most_common():
    print(f"  {k:<14}{n:>4}")

print(f"\nbest predictions:\n{'scroll':<13}{'segment':<28}{'win':>5}{'pct':>8}")
for r in ranked[:10]:
    print(f"{r['scroll']:<13}{r['segment'][:27]:<28}{int(r['windows']):>5}{float(r['pct_text']):>8.1%}")

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
order = ["speckle", "partial", "text"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].hist(score, bins=80, color="#444")
ax[0].axvline(THRESH, color="crimson", lw=2, label=f"calibrated {THRESH}")
ax[0].set_title("mass_letter over the whole corpus")
ax[0].set_xlabel("score"); ax[0].set_ylabel("windows"); ax[0].legend()

for k, c in zip(order, ("#c44", "#c94", "#4a4")):
    v = s[[i for i, t in enumerate(y) if t == k]]
    ax[1].scatter(v, order.index(k) + rng.normal(0, 0.06, len(v)), c=c, s=22, label=k)
ax[1].axvline(THRESH, color="crimson", lw=2)
ax[1].set_title("the 125 blind labels against the score they never saw")
ax[1].set_xlabel("score")
ax[1].set_yticks([0, 1, 2])
ax[1].set_yticklabels(order)
ax[1].legend()
plt.tight_layout()

The right-hand panel is the honest picture of this metric. `text` and `speckle` separate
cleanly. `partial` straddles the line, and some speckle reaches 0.75 — which is why
precision at the calibrated threshold is 0.80 and not higher.

**This is a screener, not a verdict.** It tells you where to look, not what is there. See
the limitations section of the repository README before citing any of these numbers.